In [1]:
import pandas as pd 
import duckdb 

In [2]:
conn=duckdb.connect(r"D:\SEC.gov project\database\credit_risk.db")

In [3]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def dsql(line, cell):
    df = conn.sql(cell).df()
    var_name = line.strip()
    if var_name:
        globals()[var_name] = df
    return df

Global Filters - KPI Validation

In [4]:
%%dsql 
SELECT 
*
FROM gold.fact_financial_metrics g 
ORDER BY period_end DESC 
LIMIT 1 

,accession_number,period_end,current_ratio,quick_ratio,cash_ratio,net_working_capital,interest_coverage_ratio,ocf_debt_ratio,debt_to_assets_ratio,short_term_debt_ratio,cash_earnings_conversion,fcf_to_debt_ratio,return_on_assets,filed
0,0000107687-25-000047,2025-11-30,2.69,NaN,0.63,488100000.0,NaN,0.05,0.25,0.0,4.62,NaN,0.0,2025-12-19


Industry group level consistency

In [10]:
%%dsql 
SELECT *
FROM gold.fact_financial_metrics g 
JOIN clean.num n 
ON g.accession_number=n.adsh
AND g.period_end=n.ddate 
JOIN gold.dim_filing f 
ON n.adsh=f.accession_number
WHERE LOWER(f.industry_group)='transportation equipment'
ORDER BY g.period_end DESC
LIMIT 1 

,accession_number,period_end,current_ratio,quick_ratio,cash_ratio,net_working_capital,interest_coverage_ratio,ocf_debt_ratio,debt_to_assets_ratio,short_term_debt_ratio,...,industry_code,industry_name,industry_group,filing_type,period_end_1,fiscal_year,fiscal_period,fiscal_year_end,filing_date,is_previous_report
0,0000107687-25-000047,2025-11-30,2.69,NaN,0.63,488100000.0,NaN,0.05,0.25,0.0,...,3716,MOTOR HOMES,Transportation Equipment,10-Q,2025-11-30,2026,Q1,831,2025-12-19,0


In [ ]:
%%dsql 
SELECT *
FROM gold.fact_financial_metrics g 
JOIN clean.num n 
ON g.accession_number=n.adsh
AND g.period_end=n.ddate 
JOIN gold.dim_filing f 
ON n.adsh=f.accession_number
WHERE LOWER(f.industry_group)='industrial and commercial machinery and computer equipment'
ORDER BY g.period_end DESC
LIMIT 1 

,accession_number,period_end,current_ratio,quick_ratio,cash_ratio,net_working_capital,interest_coverage_ratio,ocf_debt_ratio,debt_to_assets_ratio,short_term_debt_ratio,...,industry_code,industry_name,industry_group,filing_type,period_end_1,fiscal_year,fiscal_period,fiscal_year_end,filing_date,is_previous_report
0,0001104659-25-122321,2025-10-31,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.99,...,3523,FARM MACHINERY & EQUIPMENT,Industrial and Commercial Machinery and Comput...,10-K,2025-10-31,2025,FY,1031,2025-12-18,0


Industry level consistency 

1. Motor Homes (Transportation Equipment)

In [14]:
%%dsql 
SELECT *
FROM gold.fact_financial_metrics g 
JOIN clean.num n 
ON g.accession_number=n.adsh
AND g.period_end=n.ddate 
JOIN gold.dim_filing f 
ON n.adsh=f.accession_number
WHERE LOWER(f.industry_name)='motor homes'
ORDER BY g.period_end DESC
LIMIT 1 

,accession_number,period_end,current_ratio,quick_ratio,cash_ratio,net_working_capital,interest_coverage_ratio,ocf_debt_ratio,debt_to_assets_ratio,short_term_debt_ratio,...,industry_code,industry_name,industry_group,filing_type,period_end_1,fiscal_year,fiscal_period,fiscal_year_end,filing_date,is_previous_report
0,0000107687-25-000047,2025-11-30,2.69,NaN,0.63,488100000.0,NaN,0.05,0.25,0.0,...,3716,MOTOR HOMES,Transportation Equipment,10-Q,2025-11-30,2026,Q1,831,2025-12-19,0


Motor Vehicle & Accessories

In [17]:
%%dsql

WITH latest_period AS (

    SELECT
        f.industry_name,
        MAX(g.period_end) AS latest_period_end

    FROM gold.fact_financial_metrics g

    JOIN gold.dim_filing f
        ON g.accession_number = f.accession_number

    WHERE LOWER(f.industry_name) = 'motor vehicle parts & accessories'

    GROUP BY f.industry_name
)

SELECT
    lp.industry_name,
    lp.latest_period_end AS period_end,
    MEDIAN(g.current_ratio) AS median_current_ratio,
    MEDIAN(g.cash_ratio) AS median_cash_ratio,
    MEDIAN(g.interest_coverage_ratio) AS median_interest_coverage_ratio,
    MEDIAN(g.debt_to_assets_ratio) AS median_debt_to_assets_ratio,
    MEDIAN(g.short_term_debt_ratio) AS median_short_term_debt_ratio,
    MEDIAN(g.ocf_debt_ratio) AS median_ocf_debt_ratio,
    COUNT(g.current_ratio) AS non_null_values,
    COUNT(*) AS total_rows

FROM gold.fact_financial_metrics g

JOIN gold.dim_filing f
    ON g.accession_number = f.accession_number

JOIN latest_period lp
    ON f.industry_name = lp.industry_name
    AND g.period_end = lp.latest_period_end

GROUP BY
    lp.industry_name,
    lp.latest_period_end;

,industry_name,period_end,median_current_ratio,median_cash_ratio,median_interest_coverage_ratio,median_debt_to_assets_ratio,median_short_term_debt_ratio,median_ocf_debt_ratio,non_null_values,total_rows
0,MOTOR VEHICLE PARTS & ACCESSORIES,2025-09-30,1.95,0.22,1.495,0.085,0.045,NaN,37,37


Observed problem with multiple metric values present for the same period end, that makes sense given the grain of metric table. Thus, by statistic convention used median of the metric values of the latest period end. (Different observations were based on different fiscal periods and filing types).

Engines & Turbines

In [ ]:
%%dsql

WITH latest_period AS (

    SELECT
        f.industry_name,
        MAX(g.period_end) AS latest_period_end

    FROM gold.fact_financial_metrics g

    JOIN gold.dim_filing f
        ON g.accession_number = f.accession_number

    WHERE LOWER(f.industry_name) = 'engines & turbines'

    GROUP BY f.industry_name
)

SELECT
    lp.industry_name,
    lp.latest_period_end AS period_end,
    MEDIAN(g.current_ratio) AS median_current_ratio,
    MEDIAN(g.cash_ratio) AS median_cash_ratio,
    MEDIAN(g.interest_coverage_ratio) AS median_interest_coverage_ratio,
    MEDIAN(g.debt_to_assets_ratio) AS median_debt_to_assets_ratio,
    MEDIAN(g.short_term_debt_ratio) AS median_short_term_debt_ratio,
    MEDIAN(g.ocf_debt_ratio) AS median_ocf_debt_ratio,
    COUNT(*) AS total_rows

FROM gold.fact_financial_metrics g

JOIN gold.dim_filing f
    ON g.accession_number = f.accession_number

JOIN latest_period lp
    ON f.industry_name = lp.industry_name
    AND g.period_end = lp.latest_period_end

GROUP BY
    lp.industry_name,
    lp.latest_period_end;

,industry_name,period_end,median_current_ratio,median_cash_ratio,median_interest_coverage_ratio,median_debt_to_assets_ratio,median_short_term_debt_ratio,median_ocf_debt_ratio,non_null_values,total_rows
0,ENGINES & TURBINES,2025-09-30,1.74,0.185,2.72,0.2,0.03,NaN,6,6


Company Level Consistency

In [19]:
%%dsql

WITH latest_period AS (

    SELECT
        f.company_name,
        MAX(g.period_end) AS latest_period_end

    FROM gold.fact_financial_metrics g

    JOIN gold.dim_filing f
        ON g.accession_number = f.accession_number

    WHERE LOWER(f.company_name) like 'briggs%'

    GROUP BY f.company_name
)

SELECT
    lp.company_name,
    lp.latest_period_end AS period_end,
    MEDIAN(g.current_ratio) AS median_current_ratio,
    MEDIAN(g.cash_ratio) AS median_cash_ratio,
    MEDIAN(g.interest_coverage_ratio) AS median_interest_coverage_ratio,
    MEDIAN(g.debt_to_assets_ratio) AS median_debt_to_assets_ratio,
    MEDIAN(g.short_term_debt_ratio) AS median_short_term_debt_ratio,
    MEDIAN(g.ocf_debt_ratio) AS median_ocf_debt_ratio,
    COUNT(g.current_ratio) AS non_null_values,
    COUNT(*) AS total_rows

FROM gold.fact_financial_metrics g

JOIN gold.dim_filing f
    ON g.accession_number = f.accession_number

JOIN latest_period lp
    ON f.company_name = lp.company_name
    AND g.period_end = lp.latest_period_end

GROUP BY
    lp.company_name,
    lp.latest_period_end;

,company_name,period_end,median_current_ratio,median_cash_ratio,median_interest_coverage_ratio,median_debt_to_assets_ratio,median_short_term_debt_ratio,median_ocf_debt_ratio,non_null_values,total_rows
0,BRIGGS & STRATTON CORP,2020-03-31,0.91,0.05,-8.58,0.38,1.0,NaN,1,1


In [20]:
%%dsql

WITH latest_period AS (

    SELECT
        f.company_name,
        MAX(g.period_end) AS latest_period_end

    FROM gold.fact_financial_metrics g

    JOIN gold.dim_filing f
        ON g.accession_number = f.accession_number

    WHERE LOWER(f.company_name) like 'arcosa%'

    GROUP BY f.company_name
)

SELECT
    lp.company_name,
    lp.latest_period_end AS period_end,
    MEDIAN(g.current_ratio) AS median_current_ratio,
    MEDIAN(g.cash_ratio) AS median_cash_ratio,
    MEDIAN(g.interest_coverage_ratio) AS median_interest_coverage_ratio,
    MEDIAN(g.debt_to_assets_ratio) AS median_debt_to_assets_ratio,
    MEDIAN(g.short_term_debt_ratio) AS median_short_term_debt_ratio,
    MEDIAN(g.ocf_debt_ratio) AS median_ocf_debt_ratio,
    COUNT(g.current_ratio) AS non_null_values,
    COUNT(*) AS total_rows

FROM gold.fact_financial_metrics g

JOIN gold.dim_filing f
    ON g.accession_number = f.accession_number

JOIN latest_period lp
    ON f.company_name = lp.company_name
    AND g.period_end = lp.latest_period_end

GROUP BY
    lp.company_name,
    lp.latest_period_end;

,company_name,period_end,median_current_ratio,median_cash_ratio,median_interest_coverage_ratio,median_debt_to_assets_ratio,median_short_term_debt_ratio,median_ocf_debt_ratio,non_null_values,total_rows
0,"ARCOSA, INC.",2019-03-31,2.51,0.48,19.63,0.0,0.02,NaN,1,1


In [21]:
%%dsql

WITH latest_period AS (

    SELECT
        f.company_name,
        MAX(g.period_end) AS latest_period_end

    FROM gold.fact_financial_metrics g

    JOIN gold.dim_filing f
        ON g.accession_number = f.accession_number

    WHERE LOWER(f.company_name) like 'altenergy%'

    GROUP BY f.company_name
)

SELECT
    lp.company_name,
    lp.latest_period_end AS period_end,
    MEDIAN(g.current_ratio) AS median_current_ratio,
    MEDIAN(g.cash_ratio) AS median_cash_ratio,
    MEDIAN(g.interest_coverage_ratio) AS median_interest_coverage_ratio,
    MEDIAN(g.debt_to_assets_ratio) AS median_debt_to_assets_ratio,
    MEDIAN(g.short_term_debt_ratio) AS median_short_term_debt_ratio,
    MEDIAN(g.ocf_debt_ratio) AS median_ocf_debt_ratio,
    COUNT(g.current_ratio) AS non_null_values,
    COUNT(*) AS total_rows

FROM gold.fact_financial_metrics g

JOIN gold.dim_filing f
    ON g.accession_number = f.accession_number

JOIN latest_period lp
    ON f.company_name = lp.company_name
    AND g.period_end = lp.latest_period_end

GROUP BY
    lp.company_name,
    lp.latest_period_end;

,company_name,period_end,median_current_ratio,median_cash_ratio,median_interest_coverage_ratio,median_debt_to_assets_ratio,median_short_term_debt_ratio,median_ocf_debt_ratio,non_null_values,total_rows
0,ALTENERGY ACQUISITION CORP,2025-09-30,0.02,0.0,NaN,0.0,NaN,NaN,1,1


In [22]:
%%dsql

WITH latest_period AS (

    SELECT
        f.company_name,
        MAX(g.period_end) AS latest_period_end

    FROM gold.fact_financial_metrics g

    JOIN gold.dim_filing f
        ON g.accession_number = f.accession_number

    WHERE LOWER(f.company_name) like 'blue bird%'

    GROUP BY f.company_name
)

SELECT
    lp.company_name,
    lp.latest_period_end AS period_end,
    MEDIAN(g.current_ratio) AS median_current_ratio,
    MEDIAN(g.cash_ratio) AS median_cash_ratio,
    MEDIAN(g.interest_coverage_ratio) AS median_interest_coverage_ratio,
    MEDIAN(g.debt_to_assets_ratio) AS median_debt_to_assets_ratio,
    MEDIAN(g.short_term_debt_ratio) AS median_short_term_debt_ratio,
    MEDIAN(g.ocf_debt_ratio) AS median_ocf_debt_ratio,
    COUNT(g.current_ratio) AS non_null_values,
    COUNT(*) AS total_rows

FROM gold.fact_financial_metrics g

JOIN gold.dim_filing f
    ON g.accession_number = f.accession_number

JOIN latest_period lp
    ON f.company_name = lp.company_name
    AND g.period_end = lp.latest_period_end

GROUP BY
    lp.company_name,
    lp.latest_period_end;

,company_name,period_end,median_current_ratio,median_cash_ratio,median_interest_coverage_ratio,median_debt_to_assets_ratio,median_short_term_debt_ratio,median_ocf_debt_ratio,non_null_values,total_rows
0,BLUE BIRD CORP,2025-09-30,1.74,0.97,23.21,0.14,0.06,NaN,1,1


In [6]:
%%dsql 
SELECT DISTINCT
industry_code,
industry_group,
industry_name
FROM gold.dim_filing
ORDER BY industry_code,industry_group, industry_name

,industry_code,industry_group,industry_name
0,3510,Industrial and Commercial Machinery and Comput...,ENGINES & TURBINES
1,3523,Industrial and Commercial Machinery and Comput...,FARM MACHINERY & EQUIPMENT
2,3531,Industrial and Commercial Machinery and Comput...,CONSTRUCTION MACHINERY & EQUIP
3,3537,Industrial and Commercial Machinery and Comput...,"INDUSTRIAL TRUCKS, TRACTORS, TRAILERS & STACKERS"
4,3711,Transportation Equipment,MOTOR VEHICLES & PASSENGER CAR BODIES
5,3713,Transportation Equipment,TRUCK & BUS BODIES
6,3714,Transportation Equipment,MOTOR VEHICLE PARTS & ACCESSORIES
7,3715,Transportation Equipment,TRUCK TRAILERS
8,3716,Transportation Equipment,MOTOR HOMES
